In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
accuracy_score,
precision_score,
recall_score,
f1_score,
classification_report,
confusion_matrix'
)
]

In [9]:
df=pd.read_csv( "PhiUSIIL_Phishing_URL_Dataset.csv")

In [10]:
df.head()

,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,...,0,0,1,34,20,28,119,0,124,1
1,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,...,0,0,1,50,9,8,39,0,217,1
2,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,...,0,0,1,10,2,7,42,2,5,1
3,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,...,1,1,1,3,27,15,22,1,31,1
4,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,0.059441,...,1,0,1,244,15,34,72,1,85,1


In [12]:
df.columns

Index(['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD',
       'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb',
       'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation',
       'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL',
       'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL',
       'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
       'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS',
       'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title',
       'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots',
       'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription',
       'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet',
       'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay',
       'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS',
       'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'labe

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 55 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   URL                         235795 non-null  str    
 1   URLLength                   235795 non-null  int64  
 2   Domain                      235795 non-null  str    
 3   DomainLength                235795 non-null  int64  
 4   IsDomainIP                  235795 non-null  int64  
 5   TLD                         235795 non-null  str    
 6   URLSimilarityIndex          235795 non-null  float64
 7   CharContinuationRate        235795 non-null  float64
 8   TLDLegitimateProb           235795 non-null  float64
 9   URLCharProb                 235795 non-null  float64
 10  TLDLength                   235795 non-null  int64  
 11  NoOfSubDomain               235795 non-null  int64  
 12  HasObfuscation              235795 non-null  int64  
 13  NoOfObfuscatedChar       

In [15]:
df.isnull().sum()

URL                           0
URLLength                     0
Domain                        0
DomainLength                  0
IsDomainIP                    0
TLD                           0
URLSimilarityIndex            0
CharContinuationRate          0
TLDLegitimateProb             0
URLCharProb                   0
TLDLength                     0
NoOfSubDomain                 0
HasObfuscation                0
NoOfObfuscatedChar            0
ObfuscationRatio              0
NoOfLettersInURL              0
LetterRatioInURL              0
NoOfDegitsInURL               0
DegitRatioInURL               0
NoOfEqualsInURL               0
NoOfQMarkInURL                0
NoOfAmpersandInURL            0
NoOfOtherSpecialCharsInURL    0
SpacialCharRatioInURL         0
IsHTTPS                       0
LineOfCode                    0
LargestLineLength             0
HasTitle                      0
Title                         0
DomainTitleMatchScore         0
URLTitleMatchScore            0
HasFavic

In [17]:
 # Identify the target column
target_column = "label"
if target_column not in df.columns:
    print("\nAvailable columns:")
    print(df.columns)
    raise ValueError(
        "Target column 'label' was not found. "
        "Check the column names printed above."
    )
print("\nTarget distribution:")
print(df[target_column].value_counts())


Target distribution:
label
1    134850
0    100945
Name: count, dtype: int64


In [22]:
 # Select basic URL features
possible_features = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "ObfuscationRatio",
    "NoOfLettersInURL",
    "LetterRatioInURL",
    "NoOfDegitsInURL",
    "DegitRatioInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL"
]
# Only select features that actually exist in the CSV.
features = [col for col in possible_features if col in df.columns]
print("\nFeatures being used:")
print(features)
if len(features) == 0:
    raise ValueError(
        "None of the expected feature columns were found."
)
    


Features being used:
['URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL']


In [23]:
X = df[features].copy()
y = df[target_column].copy()

In [24]:
# Convert feature columns to numeric
X = X.apply(pd.to_numeric, errors="coerce")
# Replace infinite values
X = X.replace([np.inf, -np.inf], np.nan)
# Fill missing values with median
X = X.fillna(X.median())
# Remove rows where target is missing
valid_rows = y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

In [26]:
#Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 188636
Testing samples: 47159


In [27]:
 #Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
lr_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)


In [28]:
 #Decision Tree
dt_model = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

In [29]:
 #Evaluate Logistic Regression
print("\n" + "=" * 60)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 60)
print("Accuracy:",
      accuracy_score(y_test, lr_pred))
print("Precision:",
      precision_score(y_test, lr_pred, average="weighted"))
print("Recall:",
      recall_score(y_test, lr_pred, average="weighted"))
print("F1 Score:",
      f1_score(y_test, lr_pred, average="weighted"))
print("\nClassification Report:")
print(classification_report(y_test, lr_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_pred))



LOGISTIC REGRESSION RESULTS
Accuracy: 0.9852838270531605
Precision: 0.9855908107349385
Recall: 0.9852838270531605
F1 Score: 0.9852517114965693

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     20189
           1       0.98      1.00      0.99     26970

    accuracy                           0.99     47159
   macro avg       0.99      0.98      0.98     47159
weighted avg       0.99      0.99      0.99     47159

Confusion Matrix:
[[19523   666]
 [   28 26942]]


In [30]:
 #Evaluate Decision Tree
print("\n" + "=" * 60)
print("DECISION TREE RESULTS")
print("=" * 60)
print("Accuracy:",
      accuracy_score(y_test, dt_pred))
print("Precision:",
      precision_score(y_test, dt_pred, average="weighted"))
print("Recall:",
      recall_score(y_test, dt_pred, average="weighted"))
print("F1 Score:",
      f1_score(y_test, dt_pred, average="weighted"))
print("\nClassification Report:")
print(classification_report(y_test, dt_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, dt_pred))


DECISION TREE RESULTS
Accuracy: 0.93644903411862
Precision: 0.9408640012908241
Recall: 0.93644903411862
F1 Score: 0.9357396067851169

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.86      0.92     20189
           1       0.91      0.99      0.95     26970

    accuracy                           0.94     47159
   macro avg       0.95      0.93      0.93     47159
weighted avg       0.94      0.94      0.94     47159

Confusion Matrix:
[[17413  2776]
 [  221 26749]]


In [32]:
# Compare both models
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree"
    ],

    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, dt_pred)
    ],

    "Precision": [
        precision_score(y_test, lr_pred, average="weighted"),
        precision_score(y_test, dt_pred, average="weighted")
    ],

    "Recall": [
        recall_score(y_test, lr_pred, average="weighted"),
        recall_score(y_test, dt_pred, average="weighted")
    ],

    "F1 Score": [
        f1_score(y_test, lr_pred, average="weighted"),
        f1_score(y_test, dt_pred, average="weighted")
    ]
})

print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(results)


MODEL COMPARISON
                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.985284   0.985591  0.985284  0.985252
1        Decision Tree  0.936449   0.940864  0.936449  0.935740


In [33]:
#Find the better model
best_model = results.loc[
results["F1 Score"].idxmax(),
"Model"
]
print("\nBest model based on F1 Score:", best_model)


Best model based on F1 Score: Logistic Regression
